In [ ]:
%run scripts/verify_environment.py

verify_environment()

In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001'
idp_name_1 = 'FakeCAS' # 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
idp_name_2 = 'FakeCAS'
idp_username_2 = None
idp_password_2 = None
rdm_project_name = 'TEST-METADATA-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))
default_result_path = None
close_on_fail = False
delete_project = True
transition_timeout = 10000

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for {idp_name_2}')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_2}')
(len(idp_username_2), len(idp_password_2))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# KAKEN情報を用いた自動補完

- サブシステム名: アドオン
- ページ/アドオン: Metadata
- 機能分類: メタデータ入力
- シナリオ名: KAKEN情報による自動補完-共著者のe-Rad研究者番号を用いた補完
- 用意するテストデータ: URL一覧、アカウント1(既存ユーザー1: GRDM), アカウント2(既存ユーザー2: GRDM)
- 事前条件: 「プロジェクトに対するMetadataアドオンの登録」を実施済みであり、KAKENデータが更新済みであること

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

import scripts.metadata_v2025
importlib.reload(scripts.metadata_v2025)
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## アカウント2 ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout
    )

    # GRDMのボタンが表示されることを確認
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ログインユーザー表示をクリックする

ドロップダウンに「設定」が現れること

In [ ]:
async def _step(page):
    await page.locator(f'//a[@data-test-auth-dropdown-toggle]').click()
    await expect(page.locator(f'//a[@data-test-ad-settings]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「設定」をクリックする

「プロフィール」画面が現れること

In [ ]:
async def _step(page):
    await page.locator(f'//a[@data-test-ad-settings]').click()
    await expect(page.locator(f'//a[text() = "ID"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ID」をクリックする

「e-Rad研究者番号」画面が現れること

In [ ]:
async def _step(page):
    await page.locator(f'//a[text() = "ID"]').click()
    await expect(page.locator(f'//input[@data-bind = "value: erad"]')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

## 「あなたのe-Rad番号」に「」(空文字列)を入力する

「e-Rad」の項目が空になること

In [ ]:
async def _step(page):
    await page.locator(f'//input[@data-bind = "value: erad"]').fill('')
    await expect(page.locator(f'//button[text() = "保存" and ancestor::div[contains(@class,"active")]]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「保存」をクリックする

「設定が更新されました」と緑色で表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//button[text() = "保存" and ancestor::div[contains(@class,"active")]]').click()
    await expect(page.locator(f'//p[contains(@class, "text-success") and contains(text(), "設定が更新されました")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのURLを開き、プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前のプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
from urllib.parse import urljoin
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await page.goto(urljoin(rdm_url, 'dashboard'))
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)
        
await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること。ナビゲーションバーに表示されているURLを控えておく

In [ ]:
project_url = None

async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{rdm_project_name}"]').click()
    
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "NII Storage"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    global project_url
    project_url = page.url

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

メタデータの一覧ページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//nav//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新規メタデータ」をクリックする

スキーマ選択ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-metadata-button]').click()

    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-new-report-modal-create-report-button]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「メタデータを作成」をクリックする

メタデータ編集ウィンドウが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-report-modal-create-report-button]').click()
    
    form = ProjectMetadataForm(page)
    await expect(form.get_locator('資金配分機関情報').locator('.ember-power-select-status-icon')).to_be_attached(timeout=transition_timeout)
    await expect(form.get_locator('体系的番号におけるプログラム情報コード')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プログラム名 (日本語)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('Program name (English)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('体系的番号')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プロジェクト名 (日本語)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('Project name (English)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プロジェクトの分野').locator('.ember-power-select-status-icon')).to_be_attached(timeout=1000)

await run_pw(_step)

## 「体系的番号」をクリックする

ドロップダウンに何も現れないこと

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    locator = form.get_locator('体系的番号')
    await locator.fill('JP')
    await expect(locator).to_have_value('JP', timeout=transition_timeout)
    # e-Rad番号未設定のため、候補が表示されないこと
    time.sleep(2)
    await expect(locator.locator('xpath=..').locator('ul li')).to_have_count(0)

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//div[@class = "nav-profile-name"]').click()
    await page.locator('//button[@data-test-ad-logout]').click()

    # ログアウト成功の確認
    if idp_name_2 == 'FakeCAS':
        # FakeCASの場合はサインインボタンが表示されることを確認
        await expect(page.locator('//button[@data-test-sign-in-button]')).to_be_visible(timeout=transition_timeout)
    elif idp_name_2 is not None:
        # 通常のGakuNin DSの場合はdropdown_imgが表示されることを確認
        await expect(page.locator('//*[@id = "dropdown_img"]')).to_be_visible(timeout=transition_timeout)
    else:
        # CASの場合はログインボタンが表示されることを確認
        await expect(page.locator('//button[text() = "ログイン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アカウント1 ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    # GRDMのボタンが表示されることを確認
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ログインユーザー表示をクリックする

ドロップダウンに「設定」が現れること

In [ ]:
async def _step(page):
    await page.locator(f'//a[@data-test-auth-dropdown-toggle]').click()
    await expect(page.locator(f'//a[@data-test-ad-settings]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「設定」をクリックする

「プロフィール」画面が現れること

In [ ]:
async def _step(page):
    await page.locator(f'//a[@data-test-ad-settings]').click()
    await expect(page.locator(f'//a[text() = "ID"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ID」をクリックする

「e-Rad研究者番号」画面が現れること

In [ ]:
async def _step(page):
    await page.locator(f'//a[text() = "ID"]').click()
    await expect(page.locator(f'//input[@data-bind = "value: erad"]')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

## 「あなたのe-Rad番号」に「50735975」を入力する

「e-Rad」の項目が設定されること

In [ ]:
async def _step(page):
    await page.locator(f'//input[@data-bind = "value: erad"]').fill('50735975')
    await expect(page.locator(f'//button[text() = "保存" and ancestor::div[contains(@class,"active")]]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「保存」をクリックする

「設定が更新されました」と緑色で表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//button[text() = "保存" and ancestor::div[contains(@class,"active")]]').click()
    await expect(page.locator(f'//p[contains(@class, "text-success") and contains(text(), "設定が更新されました")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## さきほど控えたプロジェクトのURLを開く

In [ ]:
import asyncio

async def _step(page):
    await page.goto(project_url)
    await expect(page.locator('//button[text() = "アクセス権をリクエスト"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「アクセス権をリクエスト」をクリックする



In [ ]:
async def _step(page):
    await page.locator('//button[text() = "アクセス権をリクエスト"]').click()

    await expect(page.locator('//button[text() = "アクセス権がリクエストされました"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//div[@class = "nav-profile-name"]').click()
    await page.locator('//*[contains(text(), "ログアウト")]').click()

    # ログアウト成功の確認
    if idp_name_2 == 'FakeCAS':
        # FakeCASの場合はサインインボタンが表示されることを確認
        await expect(page.locator('//button[@data-test-sign-in-button]')).to_be_visible(timeout=transition_timeout)
    elif idp_name_2 is not None:
        # 通常のGakuNin DSの場合はdropdown_imgが表示されることを確認
        await expect(page.locator('//*[@id = "dropdown_img"]')).to_be_visible(timeout=transition_timeout)
    else:
        # CASの場合はログインボタンが表示されることを確認
        await expect(page.locator('//button[text() = "ログイン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アカウント2 ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout
    )

    # GRDMのボタンが表示されることを確認
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること。ナビゲーションバーに表示されているURLを控えておく

In [ ]:
project_url = None

async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{rdm_project_name}"]').click()
    
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "NII Storage"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    global project_url
    project_url = page.url

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メンバー」をクリックする

メンバーの一覧ページが表示されること

In [ ]:
async def _step(page):
    await page.locator("#projectSubnav").get_by_role("link", name="メンバー").click()

    await expect(page.locator('//h3[contains(text(), "アクセスのリクエスト")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.request-accept-button')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 権限を「読込み」に変更し、「追加」をクリックする

In [ ]:
async def _step(page):
    await page.locator("#manageAccessRequests .permissions select").select_option('read')

    await page.locator('.request-accept-button').click()
    await expect(page.locator("#manageAccessRequests .permissions select")).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

メタデータの一覧ページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//nav//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新規メタデータ」をクリックする

スキーマ選択ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-metadata-button]').click()

    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-new-report-modal-create-report-button]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「メタデータを作成」をクリックする

メタデータ編集ウィンドウが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-report-modal-create-report-button]').click()
    
    form = ProjectMetadataForm(page)
    await expect(form.get_locator('資金配分機関情報').locator('.ember-power-select-status-icon')).to_be_attached(timeout=transition_timeout)
    await expect(form.get_locator('体系的番号におけるプログラム情報コード')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プログラム名 (日本語)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('Program name (English)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('体系的番号')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プロジェクト名 (日本語)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('Project name (English)')).to_be_editable(timeout=1000)
    await expect(form.get_locator('プロジェクトの分野').locator('.ember-power-select-status-icon')).to_be_attached(timeout=1000)

await run_pw(_step)

## 「体系的番号」をクリックする

ドロップダウン中に「独立行政法人日本学術振興会」が現れること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    locator = form.get_locator('体系的番号')
    await locator.fill('JP')
    await expect(locator).to_have_value('JP', timeout=transition_timeout)
    # 候補リストが表示されること
    await expect(locator.locator('xpath=..').locator('ul li')).not_to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 最初の候補をクリックする

資金配分機関情報, 体系的番号におけるプログラム情報コード, 体系的番号, プログラム名(日本語), プロジェクト名(日本語), プロジェクトの分野が入力されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    locator = form.get_locator('体系的番号')
    # 「リアルワールドデータ」を含む候補をクリック
    await locator.locator('xpath=..').locator('ul li', has_text='リアルワールドデータ').click()

    # 各フィールドが自動入力されること
    await expect(form.get_locator('資金配分機関情報').locator('.ember-power-select-selected-item')).to_have_text('JSPS', timeout=transition_timeout)
    await expect(form.get_locator('体系的番号におけるプログラム情報コード')).to_have_value('', timeout=1000)
    await expect(form.get_locator('体系的番号')).not_to_have_value('', timeout=1000)
    await expect(form.get_locator('プログラム名 (日本語)')).to_have_value('科学研究費助成事業', timeout=1000)
    await expect(form.get_locator('Program name (English)')).to_have_value('Grants-in-Aid for Scientific Research', timeout=1000)
    await expect(form.get_locator('プロジェクト名 (日本語)')).not_to_have_value('', timeout=1000)
    await expect(form.get_locator('Project name (English)')).not_to_have_value('', timeout=1000)
    await expect(form.get_locator('プロジェクトの分野').locator('.ember-power-select-selected-item')).to_have_text('189', timeout=1000)

await run_pw(_step)

## ダッシュボードのURLを開く

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.goto(urljoin(rdm_url, 'dashboard'))
    await expect(page.locator(f'//*[@data-test-dashboard-item-title and text() = "{rdm_project_name}"]')).to_be_visible(timeout=30000)
    time.sleep(1)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{rdm_project_name}"]').click()
    
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "NII Storage"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「NII Storage」をクリックする

「メタデータ編集」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    target_storage_name = 'NII Storage'
    await grdm.get_select_expanded_storage_title_locator(page, target_storage_name).click()
    
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「新規フォルダ作成」をクリックする

フォルダ名入力テキストフィールドが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "新規フォルダ作成"]').click()
    
    await expect(page.locator('//input[@id = "createFolderInput"]')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

## 「科研費メタデータ(ENTER)」を入力する

科研費メタデータフォルダが作成されること

In [ ]:
async def _step(page):
    await page.locator('//input[@id = "createFolderInput"]').fill('科研費メタデータ')
    await page.keyboard.press('Enter')

    await expect(page.locator('//*[text() = "科研費メタデータ"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「科研費メタデータ」フォルダをクリックする

「メタデータ編集」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "科研費メタデータ"]').click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

##  「メタデータ編集」をクリックする

「メタデータ編集」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

##  「メタデータ様式」に「研究成果（公的資金対応）のメタデータ登録」を入力する

「データNo.」フィールドが表示されること

In [ ]:
async def _step(page):
    await page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select').select_option('研究成果（公的資金対応）のメタデータ登録')

    form = FileMetadataForm(page)
    await expect(form.get_locator('データ No.')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

##  「データ作成者」の「追加」をクリックする

作成者入力フィールドが表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    container = form.get_locator('データ作成者')
    panel = container.locator('tr.metadata-edit-mode')
    erad_input = panel.locator('xpath=.//label[contains(text(), "e-Rad 研究者番号")]/following-sibling::div[1]//input')

    await expect(erad_input).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

##  「e-Rad 研究者番号」に「5073」を入力する

Yusuke Komiyama が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    container = form.get_locator('データ作成者')
    panel = container.locator('tr.metadata-edit-mode')
    erad_input = panel.locator('xpath=.//label[contains(text(), "e-Rad 研究者番号")]/following-sibling::div[1]//input')
    await erad_input.fill('5073')

    await expect(page.locator('//*[contains(text(), "Yusuke Komiyama")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「Yusuke Komiyama」をクリックする

「e-Rad研究者番号」「名前 (日本語)」, 「Name (English)」が入力されること


In [ ]:
import re

async def _step(page):
    await page.locator('(//*[contains(text(), "Yusuke Komiyama")])[1]').click()

    form = FileMetadataForm(page)
    author = await form.get_author(table_label='データ作成者', row_index=0)
    # e-Rad研究者番号が入力されること
    assert author['number'] != '', f"Expected non-empty e-Rad number, got {author['number']}"
    # 名前 (日本語) が入力されること
    assert author['name_ja']['last'] != '', f"Expected non-empty last name (ja), got {author['name_ja']}"
    assert author['name_ja']['first'] != '', f"Expected non-empty first name (ja), got {author['name_ja']}"
    # Name (English) が入力されること
    assert author['name_en']['last'] != '', f"Expected non-empty last name (en), got {author['name_en']}"
    assert author['name_en']['first'] != '', f"Expected non-empty first name (en), got {author['name_en']}"

    # edit panel を閉じる
    container = form.get_locator('データ作成者')
    await container.locator('tr.metadata-edit-mode .hide-edit-row').click()

await run_pw(_step)

## データ作成者のゴミ箱アイコンをクリックする

データはありません と表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.click_table_remove_row('データ作成者', 0)
    container = form.get_locator('データ作成者')
    await expect(container.locator('//*[text() = "データはありません"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

##  「データ作成者」の「追加」をクリックする

作成者入力フィールドが表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.click_table_add_row('データ作成者')

    container = form.get_locator('データ作成者')
    panel = container.locator('tr.metadata-edit-mode')
    erad_input = panel.locator('xpath=.//label[contains(text(), "e-Rad 研究者番号")]/following-sibling::div[1]//input')
    await expect(erad_input).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

##  「名前(日本語)」に「悠介」を入力する

Yusuke Komiyama が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    container = form.get_locator('データ作成者')
    panel = container.locator('tr.metadata-edit-mode')
    ja_table = panel.locator('xpath=.//table[.//th[text()="姓"]]')
    # 「名」(first name)の入力欄に「悠介」を入力
    await ja_table.locator('tbody td:nth-of-type(3) input').fill('悠介')

    await expect(page.locator('//*[contains(text(), "50735975")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「Yusuke Komiyama」をクリックする

「e-Rad研究者番号」「名前 (日本語)」, 「Name (English)」が入力されること


In [ ]:
import re

async def _step(page):
    await page.locator('//div[contains(@class, "tt-suggestion") and contains(., "50735975")]').click()

    form = FileMetadataForm(page)
    author = await form.get_author(table_label='データ作成者', row_index=0)
    # e-Rad研究者番号が入力されること
    assert author['number'] != '', f"Expected non-empty e-Rad number, got {author['number']}"
    # 名前 (日本語) が入力されること
    assert author['name_ja']['last'] != '', f"Expected non-empty last name (ja), got {author['name_ja']}"
    assert author['name_ja']['first'] != '', f"Expected non-empty first name (ja), got {author['name_ja']}"
    # Name (English) が入力されること
    assert author['name_en']['last'] != '', f"Expected non-empty last name (en), got {author['name_en']}"
    assert author['name_en']['first'] != '', f"Expected non-empty first name (en), got {author['name_en']}"

    # edit panel を閉じる
    container = form.get_locator('データ作成者')
    await container.locator('tr.metadata-edit-mode .hide-edit-row').click()

await run_pw(_step)

## データ作成者のゴミ箱アイコンをクリックする

データはありません と表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.click_table_remove_row('データ作成者', 0)
    container = form.get_locator('データ作成者')
    await expect(container.locator('//*[text() = "データはありません"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

##  「データ作成者」の「追加」をクリックする

作成者入力フィールドが表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.click_table_add_row('データ作成者')

    container = form.get_locator('データ作成者')
    panel = container.locator('tr.metadata-edit-mode')
    erad_input = panel.locator('xpath=.//label[contains(text(), "e-Rad 研究者番号")]/following-sibling::div[1]//input')
    await expect(erad_input).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

##  「Name (English)」に「Yusuke」を入力する

Yusuke Komiyama が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    container = form.get_locator('データ作成者')
    panel = container.locator('tr.metadata-edit-mode')
    en_table = panel.locator('xpath=.//table[.//th[text()="Last Name"]]')
    # 「First Name」の入力欄に「Yusuke」を入力
    await en_table.locator('tbody td:nth-of-type(3) input').fill('Yusuke')

    await expect(page.locator('//*[contains(text(), "50735975")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「Yusuke Komiyama」をクリックする

「e-Rad研究者番号」「名前 (日本語)」, 「Name (English)」が入力されること


In [ ]:
import re

async def _step(page):
    await page.locator('//div[contains(@class, "tt-suggestion") and contains(., "50735975")]').click()

    form = FileMetadataForm(page)
    author = await form.get_author(table_label='データ作成者', row_index=0)
    # e-Rad研究者番号が入力されること
    assert author['number'] != '', f"Expected non-empty e-Rad number, got {author['number']}"
    # 名前 (日本語) が入力されること
    assert author['name_ja']['last'] != '', f"Expected non-empty last name (ja), got {author['name_ja']}"
    assert author['name_ja']['first'] != '', f"Expected non-empty first name (ja), got {author['name_ja']}"
    # Name (English) が入力されること
    assert author['name_en']['last'] != '', f"Expected non-empty last name (en), got {author['name_en']}"
    assert author['name_en']['first'] != '', f"Expected non-empty first name (en), got {author['name_en']}"

    # edit panel を閉じる
    container = form.get_locator('データ作成者')
    await container.locator('tr.metadata-edit-mode .hide-edit-row').click()

await run_pw(_step)

## 「データ管理者の種類」にて「個人」を選択する

すでに値が表示されている場合は選択不要。「個人」が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('データ管理者の種類', '個人')

    locator = form.get_locator('データ管理者の種類')
    checked = locator.locator('input[type="radio"]:checked')
    await expect(checked).to_have_value('individual', timeout=transition_timeout)

await run_pw(_step)

##  「データ管理者の e-Rad 研究者番号」に「5073」を入力する

候補 Yusuke Komiyama が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    locator = form.get_locator('データ管理者の e-Rad 研究者番号')
    await locator.fill('5073')

    await expect(page.locator('//*[contains(text(), "Yusuke Komiyama")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「KAKEN」側候補をクリックする

「データ管理者 (日本語)」「Data manager (English)」「データ管理者の所属組織名 (日本語)」が入力されること。 「Contact organization of data manager (English)」は空欄であること


In [ ]:
async def _step(page):
    await page.locator('(//*[contains(text(), "Yusuke Komiyama")])[1]').click()

    form = FileMetadataForm(page)
    # データ管理者の所属組織名 (日本語) が入力されること
    await expect(form.get_locator('データ管理者の所属組織名 (日本語)')).to_have_value(re.compile('.+'), timeout=transition_timeout)
    # Contact organization of data manager (English) は空欄であること
    await expect(form.get_locator('Contact organization of data manager (English)')).not_to_have_value(re.compile('.+'), timeout=transition_timeout)
    # データ管理機関コードは空欄
    await expect(form.get_locator('データ管理機関コード')).to_have_value('')
    # データ管理者 (日本語) が入力されること
    name_ja = await form.get_name('データ管理者 (日本語)')
    assert name_ja['last'] != '', f"Expected non-empty last name (ja), got {name_ja}"
    # Data manager (English) が入力されること
    name_en = await form.get_name('Data manager (English)')
    assert name_en['last'] != '', f"Expected non-empty last name (en), got {name_en}"

await run_pw(_step)

##  「データ管理者の e-Rad 研究者番号」「データ管理者 (日本語)」「Data manager (English)」「データ管理者の所属組織名 (日本語)」を空欄とし、「データ管理者 (日本語)」に「悠介」を入力する

候補 Yusuke Komiyama が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    # 各フィールドを空欄にする
    await form.fill('データ管理者の e-Rad 研究者番号', '')
    await form.fill('データ管理者の所属組織名 (日本語)', '')
    await form.fill('Contact organization of data manager (English)', '')
    await form.fill_name('データ管理者 (日本語)', {'last': '', 'middle': '', 'first': ''})
    await form.fill_name('Data manager (English)', {'last': '', 'middle': '', 'first': ''})

    # 「データ管理者 (日本語)」の「名」に「悠介」を入力
    await form.fill_name('データ管理者 (日本語)', {'last': '', 'middle': '', 'first': '悠介'})

    await expect(page.locator('//*[contains(text(), "50735975")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「KAKEN」側候補をクリックする

「データ管理者の e-Rad 研究者番号」「データ管理者 (日本語)」「Data manager (English)」「データ管理者の所属組織名 (日本語)」が入力されること。 「Contact organization of data manager (English)」は空欄であること


In [ ]:
async def _step(page):
    await page.locator('//div[contains(@class, "tt-suggestion") and contains(., "50735975")]').click()

    form = FileMetadataForm(page)
    # データ管理者の e-Rad 研究者番号が入力されること
    await expect(form.get_locator('データ管理者の e-Rad 研究者番号')).to_have_value(re.compile('.+'), timeout=transition_timeout)
    # データ管理者の所属組織名 (日本語) が入力されること
    await expect(form.get_locator('データ管理者の所属組織名 (日本語)')).to_have_value(re.compile('.+'), timeout=transition_timeout)
    # Contact organization of data manager (English) は空欄であること
    await expect(form.get_locator('Contact organization of data manager (English)')).not_to_have_value(re.compile('.+'), timeout=transition_timeout)
    # データ管理機関コードは空欄
    await expect(form.get_locator('データ管理機関コード')).to_have_value('')
    # データ管理者 (日本語) が入力されること
    name_ja = await form.get_name('データ管理者 (日本語)')
    assert name_ja['last'] != '', f"Expected non-empty last name (ja), got {name_ja}"
    # Data manager (English) が入力されること
    name_en = await form.get_name('Data manager (English)')
    assert name_en['last'] != '', f"Expected non-empty last name (en), got {name_en}"

await run_pw(_step)

##  「データ管理者の e-Rad 研究者番号」「データ管理者 (日本語)」「Data manager (English)」「データ管理者の所属組織名 (日本語)」を空欄とし、「Data manager (English)」に「Yusuke」を入力する

候補 Yusuke Komiyama が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    # 各フィールドを空欄にする
    await form.fill('データ管理者の e-Rad 研究者番号', '')
    await form.fill('データ管理者の所属組織名 (日本語)', '')
    await form.fill('Contact organization of data manager (English)', '')
    await form.fill_name('データ管理者 (日本語)', {'last': '', 'middle': '', 'first': ''})
    await form.fill_name('Data manager (English)', {'last': '', 'middle': '', 'first': ''})

    # 「Data manager (English)」の「First Name」に「Yusuke」を入力
    await form.fill_name('Data manager (English)', {'last': '', 'middle': '', 'first': 'Yusuke'})

    await expect(page.locator('//*[contains(text(), "50735975")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「KAKEN」側候補をクリックする

「データ管理者の e-Rad 研究者番号」「データ管理者 (日本語)」「Data manager (English)」「データ管理者の所属組織名 (日本語)」が入力されること。 「Contact organization of data manager (English)」は空欄であること


In [ ]:
async def _step(page):
    await page.locator('//div[contains(@class, "tt-suggestion") and contains(., "50735975")]').click()

    form = FileMetadataForm(page)
    # データ管理者の e-Rad 研究者番号が入力されること
    await expect(form.get_locator('データ管理者の e-Rad 研究者番号')).to_have_value(re.compile('.+'), timeout=transition_timeout)
    # データ管理者の所属組織名 (日本語) が入力されること
    await expect(form.get_locator('データ管理者の所属組織名 (日本語)')).to_have_value(re.compile('.+'), timeout=transition_timeout)
    # Contact organization of data manager (English) は空欄であること
    await expect(form.get_locator('Contact organization of data manager (English)')).not_to_have_value(re.compile('.+'), timeout=transition_timeout)
    # データ管理機関コードは空欄
    await expect(form.get_locator('データ管理機関コード')).to_have_value('')
    # データ管理者 (日本語) が入力されること
    name_ja = await form.get_name('データ管理者 (日本語)')
    assert name_ja['last'] != '', f"Expected non-empty last name (ja), got {name_ja}"
    # Data manager (English) が入力されること
    name_en = await form.get_name('Data manager (English)')
    assert name_en['last'] != '', f"Expected non-empty last name (en), got {name_en}"

await run_pw(_step)

##  「データ管理者の e-Rad 研究者番号」「データ管理者 (日本語)」「Data manager (English)」「データ管理者の所属組織名 (日本語)」を空欄とし、「データ管理者の所属組織名 (日本語)」に「国立情報学」を入力する

候補 Yusuke Komiyama が表示されること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    # 各フィールドを空欄にする
    await form.fill('データ管理者の e-Rad 研究者番号', '')
    await form.fill('データ管理者の所属組織名 (日本語)', '')
    await form.fill('Contact organization of data manager (English)', '')
    await form.fill_name('データ管理者 (日本語)', {'last': '', 'middle': '', 'first': ''})
    await form.fill_name('Data manager (English)', {'last': '', 'middle': '', 'first': ''})

    # 「データ管理者の所属組織名 (日本語)」に「国立情報学」を入力
    await form.fill('データ管理者の所属組織名 (日本語)', '国立情報学')

    await expect(page.locator('//*[contains(text(), "50735975")]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「KAKEN」側候補をクリックする

「データ管理者の e-Rad 研究者番号」「データ管理者 (日本語)」「Data manager (English)」「データ管理者の所属組織名 (日本語)」が入力されること。 「Contact organization of data manager (English)」は空欄であること


In [ ]:
async def _step(page):
    await page.locator('//div[contains(@class, "tt-suggestion") and contains(., "50735975")]').click()

    form = FileMetadataForm(page)
    # データ管理者の e-Rad 研究者番号が入力されること
    await expect(form.get_locator('データ管理者の e-Rad 研究者番号')).to_have_value(re.compile('.+'), timeout=transition_timeout)
    # データ管理者の所属組織名 (日本語) が入力されること
    await expect(form.get_locator('データ管理者の所属組織名 (日本語)')).to_have_value(re.compile('.+'), timeout=transition_timeout)
    # Contact organization of data manager (English) は空欄であること
    await expect(form.get_locator('Contact organization of data manager (English)')).not_to_have_value(re.compile('.+'), timeout=transition_timeout)
    # データ管理機関コードは空欄
    await expect(form.get_locator('データ管理機関コード')).to_have_value('')
    # データ管理者 (日本語) が入力されること
    name_ja = await form.get_name('データ管理者 (日本語)')
    assert name_ja['last'] != '', f"Expected non-empty last name (ja), got {name_ja}"
    # Data manager (English) が入力されること
    name_en = await form.get_name('Data manager (English)')
    assert name_en['last'] != '', f"Expected non-empty last name (en), got {name_en}"

await run_pw(_step)

## 「保存」をクリックする

ダイアログが非表示になること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "保存"]').click()
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select[1]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (指定がある場合) プロジェクトを削除する

1つ前のウィンドウに戻り、プロジェクトを削除する。

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await scripts.grdm.delete_project(page)
    
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

後始末

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}